In [1]:
# ===========================================================
# F1M1 — Módulo 1: Segundo caso — Farmacias
# Notebook 11 — EDA inicial del dataset de farmacia
# ===========================================================

import pandas as pd
import plotly.express as px

df = pd.read_parquet('../data/processed/farmacias_ventas_det_anonimizado.parquet')
print(f'Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}')
df.head()

Filas: 18,320,551  |  Columnas: 12


,ID_MARCA,ID_TIENDA,folio_venta,tipo_transaccion,SKU,cantidad,precio_unit,iva_unit,fecha_venta,id_empleado,costo_compra,id_cliente
0,3,1,V00010100199128,0,7501055300075,1.0,7.32,1.17,2015-09-17,176,6.3300,4ffd25e0443a
1,3,1,V00010100199138,0,7501033923234,2.0,299.31,0.00,2015-09-17,299,256.0900,25169dc93a7e
2,3,1,V00010100199138,0,7502241940945,1.0,860.16,0.00,2015-09-17,299,717.5000,25169dc93a7e
3,3,1,V00010100199138,0,7501059284500,1.0,9.50,0.00,2015-09-17,299,6.1200,25169dc93a7e
4,3,1,V00010100199138,0,7501059223905,1.0,13.01,0.00,2015-09-17,299,7.7835,25169dc93a7e


In [2]:
# Preparar datos
df['venta'] = df['precio_unit'] * df['cantidad']
df['fecha_venta'] = pd.to_datetime(df['fecha_venta'])

print(f"Rango: {df['fecha_venta'].min()} → {df['fecha_venta'].max()}")
print(f"Venta total: ${df['venta'].sum():,.0f}")
print(f"Tickets únicos: {df['folio_venta'].nunique():,}")

Rango: 2015-01-28 00:00:00 → 2016-12-31 00:00:00
Venta total: $2,745,877,787
Tickets únicos: 12,511,522


In [3]:
import sys
sys.path.insert(0, '..')
from src.kpis import ticket_promedio, share_por_categoria, pareto_80_20, cliente_activos

In [4]:
ticket_promedio(df, col_venta='venta', col_ticket='folio_venta')

39.42

In [5]:
tickets = df.groupby('folio_venta')['venta'].sum()
print(f"Mediana: ${tickets.median():,.2f}")
print(f"Promedio: ${tickets.mean():,.2f}")
print(f"Min: ${tickets.min():,.2f}")
print(f"Max: ${tickets.max():,.2f}")
print(f"\nPercentiles:")
print(tickets.describe())

Mediana: $39.42
Promedio: $219.47
Min: $-23.83
Max: $966,008.15

Percentiles:
count    1.251152e+07
mean     2.194679e+02
std      1.337977e+03
min     -2.383000e+01
25%      1.368000e+01
50%      3.942000e+01
75%      1.640300e+02
max      9.660082e+05
Name: venta, dtype: float64


In [7]:
pareto_80_20(df, col_venta='venta', col_grupo='SKU')

,SKU,venta,acumulado,es_80
8724,7501326008402,16619738.77,0.605261,True
9631,7501871721214,14221677.91,1.123190,True
10626,7502257270067,13631009.53,1.619607,True
7672,7501287624529,13547770.56,2.112993,True
2316,3594456400103,11849222.33,2.544520,True
...,...,...,...,...
61,18573,0.10,100.000000,False
1946,3282774937929,0.06,100.000000,False
10487,7502233871240,0.05,100.000000,False
93,21947,0.01,100.000000,False


In [8]:
pareto = pareto_80_20(df, col_venta='venta', col_grupo='SKU')
print(pareto[pareto['es_80'] == True].shape[0], 'productos clase A de', len(pareto), 'totales')

1692 productos clase A de 12654 totales


In [9]:
cliente_activos(df, col_cliente='id_cliente', col_fecha='fecha_venta')

,fecha_venta,id_cliente
0,2015,148
1,2016,163


In [10]:
tickets_año = df.groupby(df['fecha_venta'].dt.year)['folio_venta'].nunique().reset_index()
tickets_año.columns = ['año', 'tickets']
tickets_año

,año,tickets
0,2015,3187537
1,2016,9323985


In [11]:
print('2015:', df[df['fecha_venta'].dt.year == 2015]['fecha_venta'].min(), '→', df[df['fecha_venta'].dt.year == 2015]['fecha_venta'].max())
print('2016:', df[df['fecha_venta'].dt.year == 2016]['fecha_venta'].min(), '→', df[df['fecha_venta'].dt.year == 2016]['fecha_venta'].max())

2015: 2015-01-28 00:00:00 → 2015-12-31 00:00:00
2016: 2016-01-01 00:00:00 → 2016-12-31 00:00:00


In [12]:
print('Sucursales 2015:', df[df['fecha_venta'].dt.year == 2015]['ID_TIENDA'].nunique())
print('Sucursales 2016:', df[df['fecha_venta'].dt.year == 2016]['ID_TIENDA'].nunique())

Sucursales 2015: 138
Sucursales 2016: 146


In [14]:
df['año'] = df['fecha_venta'].dt.year
df['mes'] = df['fecha_venta'].dt.month

tickets_mes = df.groupby(['año', 'mes'])['folio_venta'].nunique().reset_index()
tickets_mes.columns = ['año', 'mes', 'tickets']

fig = px.line(tickets_mes, x='mes', y='tickets', color='año', title='Tickets por Mes', markers=True)
fig.show()

In [15]:
df = df[df['año'] == 2016]
print(f"Filas 2016: {df.shape[0]:,}")
print(f"Tickets: {df['folio_venta'].nunique():,}")
print(f"Venta total: ${df['venta'].sum():,.0f}")
print(f"Ticket promedio: ${ticket_promedio(df, col_venta='venta', col_ticket='folio_venta'):,.2f}")

Filas 2016: 13,782,639
Tickets: 9,323,985
Venta total: $2,027,547,526
Ticket promedio: $38.68
